# Übung 2: Explorative Datenanalyse -- Pandas vs. Polars
## AI Frameworks & Tools -- Block 1

**Ziel**: EDA-Workflow mit Pandas und Polars an einem realen Datensatz **selbst durchführen**.  
**Datensatz**: Berlin Airbnb Listings (Inside Airbnb, öffentlich)

---

## 0. Daten herunterladen

Der Datensatz wird automatisch von Inside Airbnb heruntergeladen.  
Beim ersten Ausführen wird die Datei geladen, danach die lokale Kopie verwendet.

In [ ]:
import pathlib
import urllib.request
import gzip
import shutil

data_dir = pathlib.Path("../data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

csv_path = data_dir / "listings_berlin.csv"
gz_path = data_dir / "listings_berlin.csv.gz"

if not csv_path.exists():
    url = "https://data.insideairbnb.com/germany/be/berlin/2025-09-23/data/listings.csv.gz"

    print(f"Lade Datensatz herunter...")
    print(f"URL: {url}")

    try:
        urllib.request.urlretrieve(url, gz_path)
        print(f"Heruntergeladen: {gz_path.stat().st_size / 1024 / 1024:.1f} MB (komprimiert)")

        with gzip.open(gz_path, 'rb') as f_in:
            with open(csv_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        gz_path.unlink()
        print(f"Entpackt: {csv_path} ({csv_path.stat().st_size / 1024 / 1024:.1f} MB)")

    except Exception as e:
        print(f"Download fehlgeschlagen: {e}")
        print("\nAlternative: Manueller Download")
        print("  1. Gehe zu http://insideairbnb.com/get-the-data/")
        print("  2. Suche 'Berlin', lade 'listings.csv.gz' (Detailed) herunter")
        print(f"  3. Entpacke und speichere als: {csv_path.resolve()}")
else:
    print(f"Datensatz bereits vorhanden: {csv_path} ({csv_path.stat().st_size / 1024 / 1024:.1f} MB)")

---
## Teil A: Geführte EDA mit Pandas (15 Min)

Die folgenden Zellen zeigen den typischen EDA-Einstieg. **Führe sie aus und verstehe den Output.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(csv_path)

print(f"Shape: {df.shape[0]:,} Zeilen x {df.shape[1]} Spalten")
print(f"\nSpalten ({len(df.columns)}):")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col} ({df[col].dtype})")
df.head(3)

In [ ]:
# Fehlende Werte
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"fehlend": missing, "prozent": missing_pct})
missing_df = missing_df[missing_df["fehlend"] > 0].sort_values("prozent", ascending=False)

print(f"Spalten mit fehlenden Werten: {len(missing_df)} von {len(df.columns)}")
print(f"Spalten mit >5%: {(missing_pct > 5).sum()}\n")
missing_df.head(15)

In [ ]:
# Duplikate
print(f"Vollständige Duplikate: {df.duplicated().sum()}")
print(f"Duplikat-IDs: {df['id'].duplicated().sum() if 'id' in df.columns else 'keine ID-Spalte'}")

In [ ]:
# Statistische Übersicht
df.describe()

### Fragen zum bisherigen Output

**Frage 1**: Wie viele Zeilen und Spalten hat der Datensatz?

*Deine Antwort*: ...

**Frage 2**: Welche 3 Spalten haben die meisten fehlenden Werte? Warum könnte das so sein?

*Deine Antwort*: ...

**Frage 3**: Gibt es Duplikate? Was würdest du damit tun?

*Deine Antwort*: ...

---
## Teil B: Eigenständige Analyse-Aufgaben (20 Min)

Ab hier schreibst du **eigenen Code**. Jede Aufgabe hat eine leere Code-Zelle. Darunter findest du einen Hint, falls du nicht weiterkommst.

### B.1 Preisspalte bereinigen

Die Spalte `price` enthält Dollar-Zeichen und Kommas (z.B. `$1,250.00`).  
**Aufgabe**: Erstelle eine neue Spalte `price_clean` die den Preis als Zahl (float) enthält.

In [ ]:
# DEIN CODE: Preisspalte bereinigen
# Erstelle df['price_clean'] als float



<details>
<summary><b>Hint B.1</b> (klicke zum Aufklappen)</summary>

Schau dir `df['price'].head()` an. Du brauchst `.str.replace()` um `$` und `,` zu entfernen, und `.astype(float)` um in eine Zahl umzuwandeln.

</details>

In [ ]:
# Prüfzelle -- führe diese aus um dein Ergebnis zu testen
assert 'price_clean' in df.columns, "Spalte 'price_clean' fehlt!"
assert df['price_clean'].dtype == float, f"Falscher Typ: {df['price_clean'].dtype}, erwartet: float"
assert df['price_clean'].notna().sum() > 0, "Alle Werte sind NaN!"
print(f"Preis-Statistik:")
print(f"  Min:    {df['price_clean'].min():.0f}")
print(f"  Max:    {df['price_clean'].max():.0f}")
print(f"  Median: {df['price_clean'].median():.0f}")
print(f"  Mean:   {df['price_clean'].mean():.0f}")
print("\nPreisspalte erfolgreich bereinigt!")

### B.2 Top-5 Bezirke nach Median-Preis

**Aufgabe**: Finde die 5 teuersten Bezirke (Spalte `neighbourhood_group_cleansed`) nach Median-Preis. Stelle das Ergebnis als **horizontales Balkendiagramm** dar.

In [ ]:
# DEIN CODE: Top-5 Bezirke + Balkendiagramm



<details>
<summary><b>Hint B.2</b> (klicke zum Aufklappen)</summary>

1. `df.groupby('neighbourhood_group_cleansed')['price_clean'].median()` gibt dir den Median pro Bezirk.  
2. `.sort_values(ascending=False).head(5)` für die Top 5.  
3. `.plot.barh()` für ein horizontales Balkendiagramm.  
4. Vergiss nicht `plt.xlabel()` und `plt.title()`.

</details>

### B.3 Eigene Visualisierung

**Aufgabe**: Formuliere eine Frage, die dich an den Daten interessiert. Beantworte sie mit einer selbst erstellten Visualisierung. **Die Frage ist der Titel des Plots.**

Beispiele für Fragen:
- Gibt es einen Zusammenhang zwischen Anzahl Reviews und Preis?
- Wie verteilen sich die Unterkunftstypen (room_type)?
- Welcher Bezirk hat die meisten Listings?

In [ ]:
# DEIN CODE: Eigene Visualisierung
# Deine Frage: ...



**Was ist deine Erkenntnis?** (1-2 Sätze)

*Deine Antwort*: ...

### B.4 Filtern und quantifizieren

**Aufgabe**: Filtere alle Listings heraus, die einen Preis über 500 **und** weniger als 10 Reviews haben. Wie viel Prozent der Daten sind das?

In [ ]:
# DEIN CODE: Filtern und Prozent berechnen



<details>
<summary><b>Hint B.4</b> (klicke zum Aufklappen)</summary>

1. Boolean Indexing: `mask = (df['price_clean'] > 500) & (df['number_of_reviews'] < 10)`  
2. Anzahl: `mask.sum()`  
3. Prozent: `mask.sum() / len(df) * 100`

</details>

**Was könnte der Grund für diese Kombination (teuer + wenig Reviews) sein?** (1-2 Sätze)

*Deine Antwort*: ...

---
## Teil C: Polars-Vergleich (10 Min)

### C.1 Aufgabe B.2 mit Polars lösen

Löse die Bezirks-Analyse (Top-5 nach Median-Preis) nochmal mit Polars.

In [ ]:
import polars as pl

df_pl = pl.read_csv(csv_path)
print(f"Polars Shape: {df_pl.shape}")

# Preis bereinigen in Polars
if df_pl.schema.get('price') == pl.Utf8:
    df_pl = df_pl.with_columns(
        pl.col('price')
        .str.replace_all(r'[\$,]', '')
        .cast(pl.Float64)
        .alias('price_clean')
    )
    pl_price = 'price_clean'
else:
    pl_price = 'price'

print("Preis bereinigt. Jetzt bist du dran:")

In [ ]:
# DEIN CODE: Top-5 Bezirke mit Polars (group_by + agg + sort)



<details>
<summary><b>Hint C.1</b> (klicke zum Aufklappen)</summary>

```python
result = df_pl.group_by('neighbourhood_group_cleansed').agg(
    pl.col('price_clean').median().alias('median_price')
).sort('median_price', descending=True).head(5)
print(result)
```

</details>

### C.2 Performance messen

In [ ]:
import time

N = 200

# Pandas
start = time.time()
for _ in range(N):
    _ = df.groupby('neighbourhood_group_cleansed')['price_clean'].median()
pandas_time = (time.time() - start) / N

# Polars
start = time.time()
for _ in range(N):
    _ = df_pl.group_by('neighbourhood_group_cleansed').agg(
        pl.col('price_clean').median()
    )
polars_time = (time.time() - start) / N

print(f'Pandas:  {pandas_time*1000:.2f} ms pro Durchlauf')
print(f'Polars:  {polars_time*1000:.2f} ms pro Durchlauf')
print(f'\nFaktor: {pandas_time / polars_time:.1f}x')

### C.3 Fazit

**AUFGABE**: Fülle die Tabelle aus (Doppelklick zum Bearbeiten):

| Aspekt | Pandas | Polars |
|--------|--------|--------|
| Syntax-Gefühl | | |
| Gemessene Performance | __ ms | __ ms |
| Ökosystem-Vorteil | | |
| Ich würde es nutzen wenn... | | |

---
## Teil D: Plenum-Vorbereitung (5 Min)

**AUFGABE**: Bereite eine 1-Minuten-Präsentation vor:
- Welche Visualisierung zeigst du? (bevorzugt deine eigene aus B.3)
- Was ist die wichtigste Erkenntnis?
- Was hat dich überrascht?

*Deine Notizen*: ...